In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import requests
import pandas as pd
import time
import os # 파일 존재 여부 확인을 위해 import

# --- 1. 준비 단계 ---

# 저장할 파일의 전체 경로 (Google Drive)
output_filename = '/content/drive/MyDrive/2025Bigdata/game2/steam_top_2_reviews.csv'

# 크롤링할 게임 ID 목록 불러오기
file_path = '/content/drive/MyDrive/2025Bigdata/game2/preprocessed_steam_data.csv'
app_ids_to_crawl = []
try:
    game_list_df = pd.read_csv(file_path)
    app_ids_to_crawl = game_list_df['appid'].unique().tolist()
    print(f"CSV 파일에서 총 {len(app_ids_to_crawl)}개의 고유 App ID를 불러왔습니다.")
except FileNotFoundError:
    print(f"❌ 오류: '{file_path}' 파일을 찾을 수 없습니다.")
except KeyError:
    print(f"❌ 오류: CSV 파일에 'app_id_column' 열이 없습니다.")


# --- ★★★ 체크포인트 기능 ★★★ ---
# 1. 이미 수집된 App ID 목록 불러오기
processed_app_ids = set()
if os.path.exists(output_filename):
    try:
        # 기존에 저장된 파일이 있으면, 'app_id' 열을 읽어 set으로 만듭니다.
        existing_df = pd.read_csv(output_filename)
        processed_app_ids = set(existing_df['app_id'].unique())
        print(f"이미 {len(processed_app_ids)}개의 게임 리뷰가 수집되었습니다. 이어서 시작합니다.")
    except pd.errors.EmptyDataError:
        print("기존 파일이 비어있습니다. 처음부터 시작합니다.")
    except Exception as e:
        print(f"기존 파일 로드 중 오류: {e}. 처음부터 시작합니다.")
else:
    print("저장된 파일이 없습니다. 처음부터 시작합니다.")
# ------------------------------------

# 수집할 리뷰 언어 설정
language_setting = 'all'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# --- 2. 크롤링 실행 ---
if app_ids_to_crawl:

    total_ids = len(app_ids_to_crawl)

    # enumerate를 사용해 현재 진행 상황(인덱스)을 함께 표시
    for i, app_id in enumerate(app_ids_to_crawl):

        # --- ★★★ 체크포인트 기능 ★★★ ---
        # 2. 이미 처리된 ID인지 확인
        if app_id in processed_app_ids:
            # 이미 처리된 ID는 건너뜁니다. (너무 많이 출력되면 시끄러우니 주석 처리)
            # print(f"[{app_id}] ({i+1}/{total_ids}) ... 이미 수집됨 (SKIP)")
            continue
        # ------------------------------------

        # (진행 상황 표시)
        print(f"▶ [{app_id}] ({i+1}/{total_ids}) 수집 시도...")

        url = f"https://store.steampowered.com/appreviews/{app_id}?json=1&language={language_setting}"

        try:
            time.sleep(3) # IP 차단 방지
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                data = response.json()

                if data and data.get('success') == 1 and 'reviews' in data:
                    top_two_reviews = data['reviews'][:2]

                    if not top_two_reviews:
                        print(f"  [{app_id}] 리뷰 데이터가 없습니다.")
                        continue

                    # --- ★★★ 즉시 저장 기능 ★★★ ---
                    # 3. 메모리의 리스트가 아닌, 즉시 저장할 리스트 생성
                    reviews_to_save = []
                    for review in top_two_reviews:
                        review_text = review.get('review')
                        if review_text:
                            reviews_to_save.append({
                                'app_id': app_id,
                                'review_text': review_text.strip()
                            })

                    # 4. 수집된 데이터를 CSV에 즉시 '추가(append)'
                    if reviews_to_save:
                        df_to_save = pd.DataFrame(reviews_to_save)

                        # 파일이 없으면 헤더(열 이름)를 쓰고, 있으면 헤더를 쓰지 않고 내용만 추가
                        write_header = not os.path.exists(output_filename)

                        df_to_save.to_csv(
                            output_filename,
                            mode='a',          # 'a' = append (추가) 모드
                            header=write_header, # 첫 저장일 때만 header 씀
                            index=False,
                            encoding='utf-8-sig'
                        )
                        print(f"✅ [{app_id}] 리뷰 {len(reviews_to_save)}개 저장 완료.")
                    # ------------------------------------

                else:
                    print(f"⚠️ [{app_id}] 리뷰를 가져오지 못했습니다.")
            else:
                print(f"❌ [{app_id}] 요청 실패. 상태 코드: {response.status_code}")

        except requests.exceptions.RequestException as e:
            print(f"❌ [{app_id}] 요청 중 예외 발생: {e}")
            # 네트워크 오류 시 잠시 대기 후 재시도 (선택적)
            time.sleep(10)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
▶ [1299170] (7501/10000) 수집 시도...
✅ [1299170] 리뷰 2개 저장 완료.
▶ [1202610] (7502/10000) 수집 시도...
✅ [1202610] 리뷰 2개 저장 완료.
▶ [1571120] (7503/10000) 수집 시도...
  [1571120] 리뷰 데이터가 없습니다.
▶ [1057740] (7504/10000) 수집 시도...
  [1057740] 리뷰 데이터가 없습니다.
▶ [632300] (7505/10000) 수집 시도...
✅ [632300] 리뷰 2개 저장 완료.
▶ [985560] (7506/10000) 수집 시도...
  [985560] 리뷰 데이터가 없습니다.
▶ [3788420] (7507/10000) 수집 시도...
  [3788420] 리뷰 데이터가 없습니다.
▶ [354200] (7508/10000) 수집 시도...
✅ [354200] 리뷰 2개 저장 완료.
▶ [2603020] (7509/10000) 수집 시도...
✅ [2603020] 리뷰 2개 저장 완료.
▶ [503160] (7510/10000) 수집 시도...
✅ [503160] 리뷰 1개 저장 완료.
▶ [3664490] (7511/10000) 수집 시도...
  [3664490] 리뷰 데이터가 없습니다.
▶ [2258510] (7512/10000) 수집 시도...
✅ [2258510] 리뷰 2개 저장 완료.
▶ [725510] (7513/10000) 수집 시도...
  [725510] 리뷰 데이터가 없습니다.
▶ [2718240] (7514/10000) 수집 시도...
✅ [2718240] 리뷰 2개 저장 완료.
▶ [3025200] (7515/10000) 수집 시도...
  [3025200] 리뷰 데이터가 없습니다.
▶ [1211980] (7516/10000) 수집 시도...
✅ [1211980] 리뷰 2개 저장 완료.
▶ [668430] (7517/10000)

리뷰데이터 없거나 한 개인 게임 아이디 출력

In [5]:
# --- 1. 파일 경로 설정 ---
# 10,000개 게임 원본 목록
game_file_path = '/content/drive/MyDrive/2025Bigdata/game2/preprocessed_steam_data.csv'
# 크롤링으로 수집된 리뷰 목록
review_file_path = '/content/drive/MyDrive/2025Bigdata/game2/steam_top_2_reviews.csv'

try:
    # --- 2. 데이터 불러오기 ---
    game_df = pd.read_csv(game_file_path)
    review_df = pd.read_csv(review_file_path)

    print(f"원본 게임 데이터: {len(game_df)} 행")
    print(f"수집된 리뷰 데이터: {len(review_df)} 행")

    if review_df.empty:
        print("\n⚠️ 리뷰 파일이 비어있습니다. 크롤링이 실행되지 않았거나 데이터가 없습니다.")

except FileNotFoundError as e:
    print(f"❌ 오류: 파일을 찾을 수 없습니다. 경로를 확인하세요.")
    print(e)
except pd.errors.EmptyDataError:
    print(f"❌ 오류: '{review_file_path}' 파일이 비어있습니다. 크롤링을 먼저 실행해주세요.")
except Exception as e:
    print(f"❌ 오류: 데이터 로드 중 문제가 발생했습니다: {e}")

else:
    # --- 3. 원본 App ID 목록 추출 ---
    source_app_id_column = 'appid'

    if source_app_id_column not in game_df.columns:
        print(f"\n❌ 중요 오류: 원본 게임 파일에 '{source_app_id_column}' 열이 없습니다.")
        print(f"  > 사용 가능한 열: {game_df.columns.tolist()}")
        print(f"  > 스크립트의 'source_app_id_column' 변수를 실제 열 이름으로 수정하세요.")
    else:
        # 원본 목록에 있는 모든 고유 App ID (Set 자료형으로)
        all_source_ids = set(game_df[source_app_id_column].unique())

        # 리뷰가 수집된 모든 고유 App ID (Set 자료형으로)
        # (이전 크롤링 스크립트에서 'app_id'로 저장했음)
        crawled_ids = set(review_df['app_id'].unique())

        # --- 4. 조건 1: 리뷰가 0개인 게임 (크롤링 안 됨) ---
        # (원본 목록에는 있지만, 수집된 목록에는 없는 ID)
        ids_with_zero_reviews = list(all_source_ids - crawled_ids)

        print("\n" + "="*40)
        print(f"📊 조건 1: 리뷰가 0개인 게임 ID (총 {len(ids_with_zero_reviews)}개)")
        print(ids_with_zero_reviews)
        print("="*40)

        # --- 5. 조건 2: 리뷰가 1개인 게임 ---
        # 'app_id'별로 몇 개의 행(리뷰)이 있는지 계산
        review_counts = review_df['app_id'].value_counts()

        # 리뷰 개수가 1개인 'app_id'만 필터링
        ids_with_one_review = list(review_counts[review_counts == 1].index)

        print(f"📊 조건 2: 리뷰가 1개인 게임 ID (총 {len(ids_with_one_review)}개)")
        print(ids_with_one_review)
        print("="*40)

원본 게임 데이터: 10000 행
수집된 리뷰 데이터: 11825 행

📊 조건 1: 리뷰가 0개인 게임 ID (총 3866개)
[np.int64(3072000), np.int64(1916930), np.int64(1753090), np.int64(2359300), np.int64(1425410), np.int64(3325960), np.int64(1835020), np.int64(1916940), np.int64(2482190), np.int64(1966100), np.int64(2359320), np.int64(1318940), np.int64(2056220), np.int64(1638430), np.int64(3194910), np.int64(1671200), np.int64(1179680), np.int64(1835040), np.int64(1548320), np.int64(1253410), np.int64(3088420), np.int64(2301990), np.int64(2113570), np.int64(3276840), np.int64(2801700), np.int64(770090), np.int64(1597480), np.int64(1949740), np.int64(3096620), np.int64(589870), np.int64(2441260), np.int64(1155120), np.int64(2089000), np.int64(1040430), np.int64(3006510), np.int64(2080820), np.int64(1572920), np.int64(1605690), np.int64(712770), np.int64(1318980), np.int64(3104840), np.int64(1482830), np.int64(983120), np.int64(1835090), np.int64(3432530), np.int64(3711060), np.int64(2515030), np.int64(745560), np.int64(1056860), n